# AIG entropy beta sweep

Study: `resnet50_aig_plus_neg_entropy_coef_lambda1em4_120ep_repeats3_ordered`.

The study is expected to contain 12 runs: beta `0.0`, `0.1`, `0.3`, `1.0`, each repeated 3 times. Plot cells below intentionally stay simple: one cell produces one chart, all in the same `plot_metric(..., interactive=True)` style. Lines show individual repeats; dashed lines show the mean for each beta.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src" / "net_complexity").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from net_complexity.studies import (
    gradient_norm_catalog,
    load_study,
    plot_metric,
)

In [ ]:
STUDY_NAME = "resnet50_aig_plus_neg_entropy_coef_lambda1em4_120ep_repeats3_ordered"
EXPECTED_BETAS = (0.0, 0.1, 0.3, 1.0)
EXPECTED_REPEATS_PER_BETA = 3
EXPECTED_RUNS = len(EXPECTED_BETAS) * EXPECTED_REPEATS_PER_BETA
STRICT_EXPECTED_SHAPE = True

# Set this manually if the server outputs are not under REPO_ROOT / "outputs/studies".
STUDY_DIR_OVERRIDE = None


def find_latest_study_dir() -> Path:
    if STUDY_DIR_OVERRIDE is not None:
        return Path(STUDY_DIR_OVERRIDE).expanduser().resolve()

    studies_root = REPO_ROOT / "outputs" / "studies"
    candidates = sorted(
        p for p in studies_root.glob(f"*{STUDY_NAME}")
        if p.is_dir() and (p / "runs").is_dir()
    )
    if not candidates:
        raise FileNotFoundError(
            f"No study matching '*{STUDY_NAME}' under {studies_root}. "
            "Set STUDY_DIR_OVERRIDE to the exact directory."
        )
    return candidates[-1]


STUDY_DIR = find_latest_study_dir()
print("REPO_ROOT:", REPO_ROOT)
print("STUDY_DIR:", STUDY_DIR)

In [ ]:
summary_df, history_df = load_study(STUDY_DIR)


def find_col(df: pd.DataFrame, candidates: tuple[str, ...]) -> str | None:
    return next((col for col in candidates if col in df.columns), None)


def as_numeric(df: pd.DataFrame, candidates: tuple[str, ...]) -> pd.Series:
    col = find_col(df, candidates)
    if col is None:
        return pd.Series(np.nan, index=df.index, dtype="float64")
    return pd.to_numeric(df[col], errors="coerce")


ACC_COL = find_col(
    history_df,
    ("valid_accuracy", "val_accuracy", "valid_acc", "val_acc", "accuracy"),
)
if ACC_COL is None:
    raise ValueError("No validation accuracy column found in history_df")

summary_df = summary_df.copy()
history_df = history_df.copy()

summary_df["entropy_beta"] = as_numeric(
    summary_df,
    (
        "label_beta",
        "model.entropy_regularization_coef",
        "mlflow.tags.entropy_regularization_coef",
    ),
)
summary_df = summary_df.sort_values(["entropy_beta", "run_name"]).reset_index(drop=True)
summary_df["repeat_idx"] = summary_df.groupby("entropy_beta", dropna=False).cumcount() + 1
summary_df["entropy_label"] = summary_df["entropy_beta"].map(lambda x: f"beta={x:g}")

run_meta = summary_df[["run_name", "entropy_beta", "entropy_label", "repeat_idx"]].set_index("run_name")
for col in run_meta.columns:
    history_df[col] = history_df["run_name"].map(run_meta[col])

if "lambda_coef" in history_df.columns:
    lambda_values = pd.to_numeric(history_df["lambda_coef"], errors="coerce")
else:
    lambda_values = pd.Series(np.nan, index=history_df.index, dtype="float64")

for split in ("train", "valid"):
    negative_entropy_col = f"{split}_negative_entropy"
    mean_p_open_col = f"{split}_mean_p_open"
    if negative_entropy_col in history_df.columns:
        negative_entropy = pd.to_numeric(history_df[negative_entropy_col], errors="coerce")
        history_df[f"{split}_entropy"] = -negative_entropy
        history_df[f"{split}_entropy_loss_term"] = history_df["entropy_beta"] * negative_entropy
    if mean_p_open_col in history_df.columns:
        mean_p_open = pd.to_numeric(history_df[mean_p_open_col], errors="coerce")
        history_df[f"{split}_lambda_p_open_loss"] = lambda_values * mean_p_open
        clipped_p_open = mean_p_open.clip(1e-8, 1.0 - 1e-8)
        history_df[f"{split}_lambda_log_odds_loss_approx"] = lambda_values * np.log(
            (1.0 - clipped_p_open) / clipped_p_open
        )
    if f"{split}_entropy_loss_term" in history_df.columns and f"{split}_lambda_p_open_loss" in history_df.columns:
        history_df[f"{split}_estimated_gate_loss"] = (
            history_df[f"{split}_lambda_p_open_loss"]
            + history_df[f"{split}_entropy_loss_term"]
        )

if "grad_norm_regularization_total_mean" in history_df.columns and "grad_norm_ce_total_mean" in history_df.columns:
    history_df["grad_ratio_regularization_to_ce_total_mean"] = (
        pd.to_numeric(history_df["grad_norm_regularization_total_mean"], errors="coerce")
        / pd.to_numeric(history_df["grad_norm_ce_total_mean"], errors="coerce").replace(0.0, np.nan)
    )

if "grad_norm_regularization_gumbel_logits_total_mean" in history_df.columns and "grad_norm_ce_gumbel_logits_total_mean" in history_df.columns:
    history_df["grad_ratio_regularization_to_ce_gumbel_logits_mean"] = (
        pd.to_numeric(history_df["grad_norm_regularization_gumbel_logits_total_mean"], errors="coerce")
        / pd.to_numeric(history_df["grad_norm_ce_gumbel_logits_total_mean"], errors="coerce").replace(0.0, np.nan)
    )

print(f"runs: {history_df['run_name'].nunique()}, history: {history_df.shape}")
print(f"accuracy column: {ACC_COL}")
display(summary_df)

In [ ]:
beta_counts = (
    summary_df.groupby("entropy_beta", dropna=False)["run_name"]
    .nunique()
    .rename("runs")
    .reset_index()
)
display(beta_counts)

if STRICT_EXPECTED_SHAPE:
    runs_loaded = int(history_df["run_name"].nunique())
    observed_betas = tuple(float(x) for x in sorted(summary_df["entropy_beta"].dropna().unique()))
    if runs_loaded != EXPECTED_RUNS:
        raise AssertionError(f"Expected {EXPECTED_RUNS} runs, got {runs_loaded}")
    if observed_betas != EXPECTED_BETAS:
        raise AssertionError(f"Expected betas {EXPECTED_BETAS}, got {observed_betas}")
    bad_counts = beta_counts[beta_counts["runs"] != EXPECTED_REPEATS_PER_BETA]
    if not bad_counts.empty:
        raise AssertionError(f"Expected {EXPECTED_REPEATS_PER_BETA} repeats per beta, got:\n{bad_counts}")

print("Study shape check passed.")

In [ ]:
def plot_beta_metric(metric: str, *, title: str | None = None, yscale: str = "linear"):
    if metric not in history_df.columns:
        raise ValueError(
            f"Column '{metric}' is missing. Available useful columns are:\n"
            + "\n".join(c for c in history_df.columns if any(k in c for k in ("accuracy", "loss", "lambda", "entropy", "active", "flops", "grad_norm", "grad_ratio", "p_open", "gate_prob")))
        )
    return plot_metric(
        history_df,
        metric,
        label_col="entropy_label",
        show_mean=True,
        interactive=True,
        yscale=yscale,
        title=title or f"{metric} / epoch by beta",
    )

## Standard graphs

These are the base checks for the run: validation accuracy, total validation loss, expected open blocks, and lambda dynamics. The open-block graph uses `valid_active_blocks_expected`, the sum of gate probabilities; it does not threshold gates at `0.5`.

In [ ]:
plot_beta_metric(ACC_COL, title="Validation accuracy by beta")

In [ ]:
plot_beta_metric("valid_loss", title="Validation loss by beta")

In [ ]:
plot_beta_metric(
    "valid_active_blocks_expected",
    title="Expected open AIG blocks by beta (sum of g_prob, no threshold)",
)

In [ ]:
plot_beta_metric("lambda_coef", title="lambda_coef by beta", yscale="log")

## Extra entropy checks

These plots test whether beta changes the loss decomposition, the gate posterior, compute, and gradient balance. They are intentionally one metric per cell so the comparison remains easy to scan. In the current implementation `valid_negative_entropy` is `-H(q)`, so for `plus_negative_entropy` the entropy contribution to the minimized loss is `beta * valid_negative_entropy = -beta * H(q)`.

In [ ]:
plot_beta_metric("valid_ce_loss", title="Validation CE loss by beta")

In [ ]:
plot_beta_metric("valid_reg_loss", title="Validation regularization term in objective by beta")

In [ ]:
plot_beta_metric("valid_regularization_loss", title="Raw AIG gate regularization by beta")

In [ ]:
plot_beta_metric("valid_mean_p_open", title="Mean posterior p(open) by beta")

In [ ]:
plot_beta_metric("valid_negative_entropy", title="Negative posterior entropy by beta")

In [ ]:
plot_beta_metric("valid_entropy", title="Posterior entropy H(q) by beta")

In [ ]:
plot_beta_metric("valid_entropy_loss_term", title="beta * negative_entropy contribution by beta")

In [ ]:
plot_beta_metric("valid_lambda_p_open_loss", title="lambda * mean_p_open contribution by beta", yscale="log")

In [ ]:
plot_beta_metric(
    "valid_lambda_log_odds_loss_approx",
    title="lambda * log((1 - p_open) / p_open) diagnostic by beta",
)

In [ ]:
plot_beta_metric("valid_estimated_gate_loss", title="Estimated gate-loss contribution by beta")

In [ ]:
plot_beta_metric("valid_mean_gate_prob", title="Mean realized gate probability by beta")

In [ ]:
plot_beta_metric("valid_aig_flops_active_ratio", title="Active FLOPs ratio by beta")

## Gradient norm checks

These show whether entropy meaningfully changes optimization pressure. Total-parameter norms answer whether the objective changed globally; gumbel-logit norms answer whether the gate parameters actually receive a different signal.

In [ ]:
plot_beta_metric("grad_norm_ce_total_mean", title="grad_norm CE total mean by beta", yscale="log")

In [ ]:
plot_beta_metric("grad_norm_regularization_total_mean", title="grad_norm regularization total mean by beta", yscale="log")

In [ ]:
plot_beta_metric("grad_norm_total_total_mean", title="grad_norm total total mean by beta", yscale="log")

In [ ]:
plot_beta_metric(
    "grad_ratio_regularization_to_ce_total_mean",
    title="grad_norm regularization / CE, total parameters by beta",
    yscale="log",
)

In [ ]:
plot_beta_metric(
    "grad_norm_ce_gumbel_logits_total_mean",
    title="grad_norm CE on gate logits by beta",
    yscale="log",
)

In [ ]:
plot_beta_metric(
    "grad_norm_regularization_gumbel_logits_total_mean",
    title="grad_norm regularization on gate logits by beta",
    yscale="log",
)

In [ ]:
plot_beta_metric(
    "grad_ratio_regularization_to_ce_gumbel_logits_mean",
    title="grad_norm regularization / CE, gate logits by beta",
    yscale="log",
)

## Available gradient norm columns

Use this table only if a metric name above is missing and the run logged a different parameter-group name.

In [ ]:
display(gradient_norm_catalog(history_df))